In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import matplotlib
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

import re

font_path = "C:/Windows/Fonts/gulim.ttc"
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc("font", family=font)

In [ ]:
team_df = pd.read_csv("Final DF.csv")
team_df

,Year,Nation,Eng_Nation,Wc_Rank,Wc_Point,Q_WR,Q_GR,F_Rank,F_Point,F_Rd,...,FS_6,FS_7,FS_8,FS_9,FS_10,FS_11,FS_12,FS_13,ATK_INDEX,DEF_INDEX
0,2002,브라질,Brazil,1,21,0.46,2.352113,1.33,818.33,0.00,...,67.95,75.27,77.05,72.73,82.09,31.95,68.95,59.61,0.141500,-0.070093
1,2002,독일,Germany,2,16,0.55,1.817073,8.67,718.33,0.00,...,63.29,66.38,67.86,72.57,80.57,37.60,62.88,63.29,0.031444,0.011481
2,2002,터키,Turkey,3,13,0.46,1.457627,33.00,593.33,0.00,...,62.65,66.45,69.80,67.05,74.05,30.85,65.18,56.78,-0.039941,0.015588
3,2002,대한민국,South Korea,4,11,0.48,2.216667,42.00,573.00,0.00,...,55.41,65.91,65.50,65.68,72.45,32.64,57.66,62.23,0.040250,0.133021
4,2002,스페인,Spain,5,11,0.65,3.227273,5.00,742.33,0.00,...,69.77,75.50,78.23,78.09,83.27,35.05,69.80,64.55,-0.105762,-0.099683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,2022,덴마크,Denmark,28,1,0.61,2.446429,13.33,1611.83,-18.00,...,68.77,72.19,70.69,67.92,70.15,17.13,66.37,53.93,NaN,NaN
188,2022,세르비아,Serbia,29,1,0.48,1.536232,31.33,1492.66,-18.00,...,62.65,66.19,72.58,69.38,64.15,18.21,64.19,51.35,NaN,NaN
189,2022,웨일스,Wales,30,1,0.44,1.233333,21.00,1545.20,10.75,...,67.38,66.19,63.88,55.19,66.27,17.62,59.62,53.12,NaN,NaN
190,2022,캐나다,Canada,31,0,0.48,2.296875,66.33,1356.74,-34.00,...,66.48,66.48,69.76,60.36,65.60,16.50,56.82,43.71,NaN,NaN


In [ ]:
len(team_df)

192

In [ ]:
import random


def make_group(df, teams_per_group=4, seed=None):
    if seed is not None:
        random.seed(seed)

    n_teams = len(df)
    n_groups = n_teams // teams_per_group
    groups = {}
    indices = df.index.tolist()
    random.shuffle(indices)

    for i in range(n_groups):
        group_name = chr(ord("A") + i)  # 'A', 'B', 'C', ...
        groups[group_name] = indices[i * teams_per_group : (i + 1) * teams_per_group]

    return groups

In [ ]:
groups = make_group(team_df[:32], seed=42)

In [ ]:
def get_match_result(home, away):
    return np.random.randint(-1, 2, 1)


get_match_result("a", "b")

array([0], dtype=int32)

In [46]:
groups

{'A': [26, 5, 10, 15],
 'B': [25, 11, 22, 6],
 'C': [19, 12, 16, 9],
 'D': [28, 14, 24, 20],
 'E': [30, 1, 13, 18],
 'F': [2, 17, 21, 3],
 'G': [29, 4, 27, 31],
 'H': [8, 23, 0, 7]}

In [ ]:
from itertools import combinations

list(combinations(groups["A"], 2))

[(26, 5), (26, 10), (26, 15), (5, 10), (5, 15), (10, 15)]

In [ ]:
from itertools import combinations


def do_groupstage(team_df, groups):
    result = {}
    if len(team_df) == 32:
        n_result = 16
    elif len(team_df) == 48:
        n_result = 32
        thrid_lst = {}
    for g in groups.keys():
        matches = {}
        for idx in groups[g]:
            matches[idx] = 0
        for home, away in list(combinations(groups[g], 2)):
            match_result = get_match_result(team_df.iloc[home], team_df.iloc[away])
            if match_result == 1:
                matches[home] += 3
            elif match_result == -1:
                matches[away] += 3
            else:
                matches[home] += 1
                matches[away] += 1
        result[g] = sorted(matches, key=lambda x: matches[x])[:3]
        if n_result == 32:
            thrid_lst[result[g][2]] = [matches[result[g][2]], g]
        result[g] = result[g][:2]
    if n_result == 32:
        sorted_lst = sorted(thrid_lst, key=lambda x: thrid_lst[x][0], reverse=True)[:8]
        for i in sorted_lst:
            result[thrid_lst[i][1]].append(i)
            pass
    return result

In [ ]:
X = team_df[:48]
groups = make_group(X)
result_group = do_groupstage(X, groups)
result_group

{'A': [13, 24, 33],
 'B': [25, 32, 44],
 'C': [29, 20, 4],
 'D': [26, 30, 16],
 'E': [12, 38, 21],
 'F': [27, 41, 1],
 'G': [10, 46, 5],
 'H': [9, 42, 36],
 'I': [31, 6],
 'J': [2, 35],
 'K': [0, 34],
 'L': [14, 11]}

In [ ]:
def do_tornament(team_df, groups):
    tornament = []
    for g in groups.keys():
        groups[g].reverse()
    return groups


do_tornament(X, result_group)

{'A': [33, 24, 13],
 'B': [44, 32, 25],
 'C': [4, 20, 29],
 'D': [16, 30, 26],
 'E': [21, 38, 12],
 'F': [1, 41, 27],
 'G': [5, 46, 10],
 'H': [36, 42, 9],
 'I': [6, 31],
 'J': [35, 2],
 'K': [34, 0],
 'L': [11, 14]}